# Final Walkthrough: Capabilities and Usage

User Workflow:

1. declare fields and gauge groups,
2. write either a local `Lagrangian(...)` or a `Model(..., lagrangian_decl=...)`,
3. call `model.lagrangian()` to have the compiled Lagrangian
4. extract a chosen vertex with `feynman_rule(...)`

Only the implemented library API is used below. The notebook defines one small display helper, `show(...)`, so that repeated vertex output is easier to read; it is not part of the library API.


In [1]:
import sys
from pathlib import Path
from fractions import Fraction

root = Path.cwd()
if not (root / "src").exists():
    root = root.parent

src_path = root / "src"
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from symbolica import S, Expression
from symbolic.vertex_engine import I
from symbolic.spenso_structures import (
    gauge_generator,
    structure_constant,
    weak_gauge_generator,
    weak_structure_constant,
)
from model import (
    CovD,
    Field,
    FieldStrength,
    Gamma,
    GaugeFixing,
    GaugeGroup,
    GaugeRepresentation,
    GhostLagrangian,
    IndexType,
    Lagrangian,
    Metric,
    Model,
    PartialD,
)

mu = S("mu")
nu = S("nu")

lam4 = S("lam4")
y = S("y")
gBox = S("gBox")
e = S("e")
gs = S("gs")
g1 = S("g1")
g2 = S("g2")
Qe = S("Qe")
YL = S("YL")
YR = S("YR")
YH = S("YH")
xi = S("xi")


def show(title, result):
    print(title)
    if isinstance(result, dict):
        print(f"{len(result)} vertex signature(s)")
        print()
        for signature, expression in result.items():
            print("Vertex:", signature)
            print("Rule:", expression)
            print()
    else:
        print(result)
        print()


print("Python:", sys.executable)
print("Repository root:", root.resolve())


Python: /Users/rems/Documents/MScThesis/.venv/bin/python
Repository root: /Users/rems/Documents/MScThesis


## 1. Index declarations

The walkthrough builds the standard `IndexType` objects locally from Spenso `Representation` values instead of importing `SPINOR_INDEX`, `LORENTZ_INDEX`, and the gauge/flavor index constants from `model`.

Each `IndexType` pairs a human-readable name with:
- a Spenso `representation` (what tensor slot family to use),
- a project `kind` string (how labels are keyed through lowering and the vertex engine),
- an optional `prefix` for default label names.

For spinor and Lorentz indices, the `kind` string is yours to choose as long as it is used consistently across the model. The engine resolves those slots from the Spenso representation (`bis(4)`, `mink(4)`), not from hard-coded `"spinor"` / `"lorentz"` literals. Gauge and flavor indices still need distinct `kind` strings when their representations coincide (for example `cof(3)` for both color and generation).

In [2]:
from symbolica.community.spenso import Representation

SPINOR_KIND = "spinor"
LORENTZ_KIND = "lorentz"
COLOR_FUND_KIND = "color_fund"
COLOR_ADJ_KIND = "color_adj"
WEAK_FUND_KIND = "weak_fund"
WEAK_ADJ_KIND = "weak_adj"

SPINOR_INDEX = IndexType("Spinor", Representation.bis(4), SPINOR_KIND, prefix="i")
LORENTZ_INDEX = IndexType("Lorentz", Representation.mink(4), LORENTZ_KIND, prefix="mu")
COLOR_FUND_INDEX = IndexType("ColorFund", Representation.cof(3), COLOR_FUND_KIND, prefix="c")
COLOR_ADJ_INDEX = IndexType("ColorAdj", Representation.coad(8), COLOR_ADJ_KIND, prefix="a")
WEAK_FUND_INDEX = IndexType("WeakFund", Representation.cof(2), WEAK_FUND_KIND, prefix="w")
WEAK_ADJ_INDEX = IndexType("WeakAdj", Representation.coad(3), WEAK_ADJ_KIND, prefix="aw")

for index in (
    SPINOR_INDEX,
    LORENTZ_INDEX,
    COLOR_FUND_INDEX,
    COLOR_ADJ_INDEX,
    WEAK_FUND_INDEX,
    WEAK_ADJ_INDEX,
):
    print(f"{index.name:10} kind={index.kind:12} rep={index.representation}")

Spinor     kind=spinoR       rep=bis(4)
Lorentz    kind=lorentzz     rep=mink(4)
ColorFund  kind=color_fund   rep=cof(3)
ColorAdj   kind=color_adj    rep=coad(8)
WeakFund   kind=weak_fund    rep=cof(2)
WeakAdj    kind=weak_adj     rep=coad(3)


## 2. Field Declaration

The framework supports the field types used below:

- real scalars,
- complex scalars,
- Dirac fermions,
- gauge bosons,
- ghost fields.

In [3]:
Phi = Field("Phi", spin=0, self_conjugate=True, symbol=S("phi"))
PhiC = Field(
    "PhiC",
    spin=0,
    self_conjugate=False,
    symbol=S("phiC"),
    conjugate_symbol=S("phiCdag"),
)
Psi = Field(
    "Psi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psi"),
    conjugate_symbol=S("psibar"),
    indices=(SPINOR_INDEX,),
)
Photon = Field(
    "A",
    spin=1,
    self_conjugate=True,
    symbol=S("A"),
    indices=(LORENTZ_INDEX,),
)
Gluon = Field(
    "G",
    spin=1,
    self_conjugate=True,
    symbol=S("G"),
    indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX),
)
GhostGluon = Field(
    "ghG",
    spin=0,
    kind="ghost",
    self_conjugate=False,
    symbol=S("ghG"),
    conjugate_symbol=S("ghGbar"),
    indices=(COLOR_ADJ_INDEX,),
)

Electron = Field(
    "e",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("e"),
    conjugate_symbol=S("ebar"),
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Q": Qe},
)
Quark = Field(
    "q",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("q"),
    conjugate_symbol=S("qbar"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
)
W = Field(
    "W",
    spin=1,
    self_conjugate=True,
    symbol=S("W"),
    indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX),
)
B = Field(
    "B",
    spin=1,
    self_conjugate=True,
    symbol=S("B"),
    indices=(LORENTZ_INDEX,),
)
LDoublet = Field(
    "L",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("L"),
    conjugate_symbol=S("Lbar"),
    indices=(SPINOR_INDEX, WEAK_FUND_INDEX),
    quantum_numbers={"Y": YL},
)
ERight = Field(
    "ER",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("ER"),
    conjugate_symbol=S("ERbar"),
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Y": YR},
)
Higgs = Field(
    "H",
    spin=0,
    self_conjugate=False,
    symbol=S("H"),
    conjugate_symbol=S("Hdag"),
    indices=(WEAK_FUND_INDEX,),
    quantum_numbers={"Y": YH},
)


A few points are worth noting immediately:

- complex fields carry both `symbol` and `conjugate_symbol`,
- fermions carry a spinor index,
- gauge bosons carry a Lorentz index and, for non-abelian groups, an adjoint gauge index,
- ghosts are declared explicitly with `kind="ghost"`.

The conjugated version of a non-self-conjugate field is written with `.bar`, for example `Psi.bar`, `Electron.bar`, or `Higgs.bar`.

## 3. Gauge Groups, Representations, and Charges

In [4]:
COLOR_FUND_REP = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fundamental",
)
WEAK_DOUBLET_REP = GaugeRepresentation(
    index=WEAK_FUND_INDEX,
    generator_builder=weak_gauge_generator,
    name="doublet",
)

U1QED = GaugeGroup(
    name="U1QED",
    abelian=True,
    coupling=e,
    gauge_boson=Photon,
    charge="Q",
)
SU3C = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gs,
    gauge_boson=Gluon,
    ghost_field=GhostGluon,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)
SU2L = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=g2,
    gauge_boson=W,
    structure_constant=weak_structure_constant,
    representations=(WEAK_DOUBLET_REP,),
)
U1Y = GaugeGroup(
    name="U1Y",
    abelian=True,
    coupling=g1,
    gauge_boson=B,
    charge="Y",
)

{
    "U1QED": {
        "abelian": U1QED.abelian,
        "coupling": U1QED.coupling,
        "gauge boson": U1QED.gauge_boson.name,
        "charge label": U1QED.charge,
    },
    "SU3C": {
        "abelian": SU3C.abelian,
        "coupling": SU3C.coupling,
        "gauge boson": SU3C.gauge_boson.name,
        "representations": tuple(rep.name for rep in SU3C.representations),
    },
    "SU2L x U1Y": {
        "SU2L boson": SU2L.gauge_boson.name,
        "SU2L representation": tuple(rep.name for rep in SU2L.representations),
        "U1Y boson": U1Y.gauge_boson.name,
        "U1Y charge label": U1Y.charge,
    },
}

{'U1QED': {'abelian': True,
  'coupling': e,
  'gauge boson': 'A',
  'charge label': 'Q'},
 'SU3C': {'abelian': False,
  'coupling': gs,
  'gauge boson': 'G',
  'representations': ('fundamental',)},
 'SU2L x U1Y': {'SU2L boson': 'W',
  'SU2L representation': ('doublet',),
  'U1Y boson': 'B',
  'U1Y charge label': 'Y'}}

In [5]:
{
    "electron charge entry": Electron.quantum_numbers,
    "quark color representation": SU3C.matter_representation(Quark).name,
    "left doublet weak representation": SU2L.matter_representation(LDoublet).name,
    "left doublet hypercharge entry": LDoublet.quantum_numbers["Y"],
    "Higgs hypercharge entry": Higgs.quantum_numbers["Y"],
}

{'electron charge entry': {'Q': Qe},
 'quark color representation': 'fundamental',
 'left doublet weak representation': 'doublet',
 'left doublet hypercharge entry': YL,
 'Higgs hypercharge entry': YH}

## 4. Building Models

Two user-facing entry points are available:

- `Lagrangian(...)` for operators built from fields, `PartialD(...)`, `Gamma(...)`, and tensor factors such as `Metric(...)`,
- `Model(..., lagrangian_decl=...)` for declarations that need more stuff like `CovD(...)`, `FieldStrength(...)`, `GaugeFixing(...)`, and `GhostLagrangian(...)`.

In [6]:
local_scalar_lagrangian = Lagrangian(lam4 * Phi * Phi * Phi * Phi)

qed_current_model = Model(
    gauge_groups=(U1QED,),
    fields=(Electron, Photon),
    lagrangian_decl=I * Electron.bar * Gamma(mu) * CovD(Electron, mu),
)

print("local source term:   ",local_scalar_lagrangian)
print("gauge-aware source term:     ",qed_current_model.lagrangian_decl)

local source term:    lam4 * Phi * Phi * Phi * Phi
gauge-aware source term:      1𝑖 * e.bar * Gamma(mu) * CovD(e, mu)


The object returned by `model.lagrangian()` is the compiled extraction object. That is the object on which you call `feynman_rule(...)`.

## 5. Lagrangian Examples

This section is a compact catalog of interactions that are already supported. The examples use `show(...)` for readable output.


### Scalar Interaction

In [7]:
scalar_lagrangian = Lagrangian(lam4 * Phi * Phi * Phi * Phi)

show(
    "Scalar vertex Gamma(Phi, Phi, Phi, Phi)",
    scalar_lagrangian.feynman_rule(Phi, Phi, Phi, Phi, simplify=True),
)
show(
    "Scalar vertices discovered by feynman_rule()",
    scalar_lagrangian.feynman_rule(simplify=True),
)


Scalar vertex Gamma(Phi, Phi, Phi, Phi)
24𝑖*lam4

Scalar vertices discovered by feynman_rule()
1 vertex signature(s)

Vertex: ('Phi', 'Phi', 'Phi', 'Phi')
Rule: 24𝑖*lam4



### Fermion Interaction

In [8]:
fermion_lagrangian = Lagrangian(y * Psi.bar * Psi * Phi)

show(
    "Yukawa vertex (Psi.bar, Psi, Phi)",
    fermion_lagrangian.feynman_rule(Psi.bar, Psi, Phi, simplify=True),
)


TypeError: Unknown type: NoneType

The bispinor metric contracts the two spinor legs. This is the expected momentum-space Yukawa vertex, with the usual overall `i` and delta function.

### Derivative Interaction

Nested `PartialD(...)` factors are supported for local derivative operators. Here the Lagrangian contains both `\phi^4` and `\phi^3 \, \partial^2 \phi`; the two contributions are merged into the same four-scalar vertex.


In [ ]:
derivative_lagrangian = Lagrangian(
    lam4 * Phi * Phi * Phi * Phi,
    gBox * Phi * Phi * Phi * PartialD(PartialD(Phi, mu), nu) * Metric(mu, nu),
)

show(
    "Scalar vertex with local and derivative contributions",
    derivative_lagrangian.feynman_rule(Phi, Phi, Phi, Phi, simplify=True),
)


Scalar vertex with local and derivative contributions
24𝑖*lam4-6𝑖*gBox*pcomp(q1,mu1_int)^2-6𝑖*gBox*pcomp(q2,mu1_int)^2-6𝑖*gBox*pcomp(q3,mu1_int)^2-6𝑖*gBox*pcomp(q4,mu1_int)^2



The derivative term contributes a sum of `q_1^2 + q_2^2 + q_3^2 + q_4^2`. That is exactly what one expects: in a symmetric `\phi^3 \partial^2 \phi` operator, any one of the four external scalar legs can be the differentiated field.

### Gauge Interaction

Pure gauge sectors are declared with `FieldStrength(...)`. The same compiled Lagrangian can extract one chosen Yang-Mills vertex or enumerate all generated pure-gauge signatures.

In [ ]:
yang_mills_model = Model(
    gauge_groups=(SU3C,),
    fields=(Gluon,),
    lagrangian_decl=(
        -(Expression.num(1) / Expression.num(4))
        * FieldStrength(SU3C, mu, nu)
        * FieldStrength(SU3C, mu, nu)
    ),
)
yang_mills_lagrangian = yang_mills_model.lagrangian()

show(
    "Yang-Mills vertex ",
    yang_mills_lagrangian.feynman_rule( simplify=True),
)


Yang-Mills vertex 
3 vertex signature(s)

Vertex: ('G', 'G')
Rule: 1𝑖*g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)-1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu2)*pcomp(q2,mu1)

Vertex: ('G', 'G', 'G')
Rule: gs*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu3)-gs*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q2,mu3)-gs*g(mink(4, mu1),mink(4, mu3))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu2)+gs*g(mink(4, mu1),mink(4, mu3))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q3,mu2)+gs*g(mink(4, mu2),mink(4, mu3))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q2,mu1)-gs*g(mink(4, mu2),mink(4, mu3))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q3,mu1)

Vertex: ('G', 'G', 'G', 'G')
Rule: -1𝑖*gs^2*g(mink(4, mu1),mink(4, mu2))*g(mink(4, mu3),mink(4, mu4))*f(coad(8, color_adj_mid_G_SU3C),coad(8, a1),coad(8, a3))*f(coad(8, color_adj_mid_G_SU3C),coad(8, a2),coad(8, a4))-1𝑖*gs^2*g(mink(4, mu1)

This output is deliberately explicit: Lorentz metrics, momenta, and the non-abelian structure constant are all shown. It is the expected three-gluon Yang-Mills vertex written in the codebase's current symbolic conventions.

### Ghost Interaction

The non-abelian ghost sector is exposed through `GhostLagrangian(...)` once the gauge group knows its ghost field.

In [ ]:
ghost_model = Model(
    gauge_groups=(SU3C,),
    fields=(Gluon, GhostGluon),
    lagrangian_decl=GhostLagrangian(SU3C),
)
ghost_lagrangian = ghost_model.lagrangian()

show(
    "Ghost-gluon vertex Gamma(ghG.bar, G, ghG)",
    ghost_lagrangian.feynman_rule(GhostGluon.bar, Gluon, GhostGluon, simplify=True),
)


Ghost-gluon vertex Gamma(ghG.bar, G, ghG)
gs*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu2)



The cubic antighost-gluon-ghost vertex carries the antighost momentum. That is the correct momentum dependence for the ordinary Faddeev-Popov ghost term in integrated form.

### QED

For abelian gauge interactions, one declares the charge on the field and writes the kinetic term with `CovD(...)`.

In [ ]:
qed_model = Model(
    gauge_groups=(U1QED,),
    fields=(Electron, Photon),
    lagrangian_decl=I * Electron.bar * Gamma(mu) * CovD(Electron, mu),
)
qed_lagrangian = qed_model.lagrangian()

show(
    "QED vertex Gamma(e.bar, e, A)",
    qed_lagrangian.feynman_rule(Electron.bar, Electron, Photon, simplify=True),
)


QED vertex Gamma(e.bar, e, A)
1𝑖*e*Qe*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))



This is the expected abelian current vertex: a gamma matrix times the charge factor `Qe` and the coupling `e`. The sign convention follows the codebase's covariant-derivative conventions.

### QCD

For non-abelian matter, the field carries a color index and the gauge group supplies the generator structure.

In [ ]:
qcd_model = Model(
    gauge_groups=(SU3C,),
    fields=(Quark, Gluon),
    lagrangian_decl=I * Quark.bar * Gamma(mu) * CovD(Quark, mu),
)
qcd_lagrangian = qcd_model.lagrangian()

show(
    "QCD quark-gluon vertex Gamma(q.bar, q, G)",
    qcd_lagrangian.feynman_rule(Quark.bar, Quark, Gluon, simplify=True),
)


QCD quark-gluon vertex Gamma(q.bar, q, G)
1𝑖*gs*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))



Compared with QED, the new ingredient is the generator `t(...)` acting on the color indices. That is the standard non-abelian color structure of the quark-gluon vertex.

### Mixing the Declarative API with Raw Spenso Tensors

So far every Dirac structure has been spelled with the declarative wrapper `Gamma(mu)`. Under the hood it emits an ordinary Spenso `gamma(...)` tensor, so a Lagrangian can equally well be written directly against the upstream Spenso API.  

The type chain is short:

- `TensorName.gamma()` is a `TensorName` (the bare tensor head, no slots).
- Calling it with typed slots, e.g. `gamma(bis(4, i), bis(4, j), mink(4, mu))`, returns a `TensorIndices`.
- `TensorIndices` auto-coerces to a Symbolica `Expression` whenever it meets one through `*`, `+` or `-`, so we can use it directly in `lagrangian_decl=...` without an explicit `.to_expression()` call. Project helpers like `gamma_matrix(...)` only call `.to_expression()` so that they always hand back a guaranteed `Expression`.

Two requirements have to be respected when bypassing the declarative wrappers:

1. The Spenso tensor must be inserted between the conjugated and unconjugated fermion factors, exactly where `Gamma(...)` would sit.
2. The spinor labels appearing in the Spenso expression must be propagated to the surrounding `FieldOccurrence`s with `index_labels={SPINOR_KIND: ...}`. The wrappers do this automatically; raw Spenso input does not.


In [ ]:
from symbolica.community.spenso import Representation, TensorName
from model import SPINOR_KIND

BIS = Representation.bis(4)
MINK = Representation.mink(4)
gamma = TensorName.gamma()
gamma5 = TensorName.gamma5()

gV = S("gV")
i_s, j_s, k_s = S("i_s"), S("j_s"), S("k_s")

declarative_yukawa = Model(
    fields=(Psi, Phi),
    lagrangian_decl=gV * Psi.bar * Gamma(mu) * Psi * Phi,
).lagrangian()

spenso_yukawa = Model(
    fields=(Psi, Phi),
    lagrangian_decl=gV
    * Psi.bar(index_labels={SPINOR_KIND: i_s})
    * gamma(BIS(i_s), BIS(j_s), MINK(mu))
    * Psi(index_labels={SPINOR_KIND: j_s})
    * Phi,
).lagrangian()

show(
    "Declarative Gamma(mu):",
    declarative_yukawa.feynman_rule(Psi.bar, Psi, Phi, simplify=True, include_delta=False),
)
show(
    "Bare Spenso TensorName.gamma() with explicit spinor labels:",
    spenso_yukawa.feynman_rule(Psi.bar, Psi, Phi, simplify=True, include_delta=False),
)

Declarative Gamma(mu):
1𝑖*gV*gamma(bis(4, i1),bis(4, i2),mink(4, mu))

Bare Spenso TensorName.gamma() with explicit spinor labels:
1𝑖*gV*gamma(bis(4, i1),bis(4, i2),mink(4, mu))



The two forms produce identical vertices, which confirms that bare Spenso input is a first-class building block of the declarative API.

The mechanism generalises to any Spenso-valued bispinor structure. The next cell builds a left-handed chiral current, $\frac{1}{2}\bar\psi\,\gamma^\mu(1-\gamma_5)\,\psi$, straight from `TensorName.gamma()` and `TensorName.gamma5()` (no project helpers, no declarative wrapper). It then defines a custom Spenso tensor `my_dirac_struct` from scratch via `TensorName("...")` and uses it in the same slot, to show that *any* tensor head with typed spinor/Lorentz slots is a valid Dirac block.

Finally, a short cautionary example shows what happens when the spinor labels are *not* propagated. 

In [ ]:
HALF = Expression.num(1) / Expression.num(2)
left_chiral = (
    gamma(BIS(i_s), BIS(j_s), MINK(mu))
    - gamma(BIS(i_s), BIS(k_s), MINK(mu)) * gamma5(BIS(k_s), BIS(j_s))
)

chiral_yukawa = Model(
    fields=(Psi, Phi),
    lagrangian_decl=gV
    * Psi.bar(index_labels={SPINOR_KIND: i_s})
    * (HALF * left_chiral)
    * Psi(index_labels={SPINOR_KIND: j_s})
    * Phi,
).lagrangian()

show(
    "Chiral current (1 - gamma5) / 2, built from TensorName.gamma() / .gamma5():",
    chiral_yukawa.feynman_rule(Psi.bar, Psi, Phi, simplify=True, include_delta=False),
)

# Custom Spenso tensor head defined from scratch (any name, typed slots).
my_dirac_struct = TensorName("my_dirac_struct")

custom_yukawa = Model(
    fields=(Psi, Phi),
    lagrangian_decl=gV
    * Psi.bar(index_labels={SPINOR_KIND: i_s})
    * my_dirac_struct(BIS(i_s), BIS(j_s), MINK(mu))
    * Psi(index_labels={SPINOR_KIND: j_s})
    * Phi,
).lagrangian()

show(
    "Custom TensorName('my_dirac_struct') used in the same slot:",
    custom_yukawa.feynman_rule(Psi.bar, Psi, Phi, simplify=True, include_delta=False),
)

# Cautionary example: the same Spenso input without explicit spinor labels.
broken_yukawa = Model(
    fields=(Psi, Phi),
    lagrangian_decl=gV
    * Psi.bar
    * gamma(BIS(i_s), BIS(j_s), MINK(mu))
    * Psi
    * Phi,
).lagrangian()

show(
    "Wrong: bare Spenso gamma without label propagation (extra metric + floating i_s, j_s):",
    broken_yukawa.feynman_rule(Psi.bar, Psi, Phi, simplify=True, include_delta=False),
)

Chiral current (1 - gamma5) / 2, built from TensorName.gamma() / .gamma5():
1𝑖/2*gV*gamma(bis(4, i1),bis(4, i2),mink(4, mu))-1𝑖/2*gV*gamma(bis(4, i1),bis(4, k_s),mink(4, mu))*gamma5(bis(4, k_s),bis(4, i2))

Custom TensorName('my_dirac_struct') used in the same slot:
1𝑖*gV*my_dirac_struct(bis(4, i1),bis(4, i2),mink(4, mu))

Wrong: bare Spenso gamma without label propagation (extra metric + floating i_s, j_s):
1𝑖*gV*g(bis(4, i1),bis(4, i2))*gamma(bis(4, i_s),bis(4, j_s),mink(4, mu))



### Full SU(2) x U(1) Electroweak / SM-Style Sector

The next example combines a left-handed lepton doublet, a right-handed singlet, and a Higgs doublet in one unbroken electroweak model. This is a realistic SM-style slice of the supported API.

In [ ]:
electroweak_model = Model(
    gauge_groups=(SU2L, U1Y),
    fields=(LDoublet, ERight, Higgs, W, B),
    lagrangian_decl=(
        I * LDoublet.bar * Gamma(mu) * CovD(LDoublet, mu)
        + I * ERight.bar * Gamma(mu) * CovD(ERight, mu)
        + CovD(Higgs.bar, mu) * CovD(Higgs, mu)
    ),
)
electroweak_lagrangian = electroweak_model.lagrangian()

print(
    "Electroweak current Gamma(L.bar, L, W)\n",
    electroweak_lagrangian.feynman_rule(LDoublet.bar, LDoublet, W, simplify=True),
)
print(
    "Hypercharge current Gamma(ER.bar, ER, B)\n ",
    electroweak_lagrangian.feynman_rule(ERight.bar, ERight, B, simplify=True),
)
print(
    "Mixed Higgs contact Gamma(H.bar, H, W, B)  \n",
    electroweak_lagrangian.feynman_rule(Higgs.bar, Higgs, W, B, simplify=True),
)

print("=============================================================")

show(
    "Electroweak",
    electroweak_lagrangian.feynman_rule( simplify=True),
)

Electroweak current Gamma(L.bar, L, W)
 1𝑖*g2*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(3, aw3),cof(2, w1),cof(2, w2))
Hypercharge current Gamma(ER.bar, ER, B)
  1𝑖*g1*YR*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))
Mixed Higgs contact Gamma(H.bar, H, W, B)  
 2𝑖*g1*g2*YH*g(mink(4, mu3),mink(4, mu4))*t(coad(3, aw3),cof(2, w1),cof(2, w2))
Electroweak
11 vertex signature(s)

Vertex: ('L.bar', 'L', 'W')
Rule: 1𝑖*g2*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(3, aw3),cof(2, w1),cof(2, w2))

Vertex: ('L.bar', 'L', 'B')
Rule: 1𝑖*g1*YL*g(cof(2, w1),cof(2, w2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))

Vertex: ('L.bar', 'L')
Rule: 1𝑖*g(cof(2, w1),cof(2, w2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu1_int))*pcomp(q2,mu1_int)

Vertex: ('ER.bar', 'ER', 'B')
Rule: 1𝑖*g1*YR*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))

Vertex: ('ER.bar', 'ER')
Rule: 1𝑖*gamma(bis(4, i1),bis(4, i2),mink(4, mu1_int))*pcomp(q2,mu1_int)

Vertex: ('H.bar', 'H', 'W')
Rule: -1𝑖*g2*t(coad(3, aw3),cof(2, w1),cof(2, w2))*

This one model already shows several distinct capabilities:

- the SU(2) current of a weak doublet,
- the U(1) hypercharge current of a singlet,
- the mixed Higgs-gauge contact term from `(D_\mu H)^\dagger (D^\mu H)`.

That is exactly the kind of electroweak structure one expects from the implemented `CovD(...)` lowering.

### Distinguishing Repeated SU(2) Factors by Representation Index

This diagnostic model uses three independent SU(2) gauge groups, `SU2A x SU2B x SU2C`. The point is not phenomenology; it is to stress-test whether `CovD(...)` matches each field to the correct gauge group through its specific representation index, even when all three groups have the same underlying type.

In [ ]:
from symbolica.community.spenso import Representation, TensorName

from model import IndexType


gA = S("gA")
gB = S("gB")
gC = S("gC")


SU2A_FUND = IndexType("SU2A_FUND", Representation.cof(2), "su2a_fund", prefix="a")
SU2A_ADJ = IndexType("SU2A_ADJ", Representation.coad(3), "su2a_adj", prefix="aa")
SU2B_FUND = IndexType("SU2B_FUND", Representation.cof(2), "su2b_fund", prefix="b")
SU2B_ADJ = IndexType("SU2B_ADJ", Representation.coad(3), "su2b_adj", prefix="ab")
SU2C_FUND = IndexType("SU2C_FUND", Representation.cof(2), "su2c_fund", prefix="c")
SU2C_ADJ = IndexType("SU2C_ADJ", Representation.coad(3), "su2c_adj", prefix="ac")


def make_su2_generator(adj_index, matter_index):
    def build(adj_label, left_label, right_label):
        return TensorName.t()(
            adj_index.representation(adj_label),
            matter_index.representation(left_label),
            matter_index.representation(right_label),
        ).to_expression()

    return build


def make_su2_structure_constant(adj_index):
    def build(a, b, c):
        return TensorName.f()(
            adj_index.representation(a),
            adj_index.representation(b),
            adj_index.representation(c),
        ).to_expression()

    return build


epsA = make_su2_structure_constant(SU2A_ADJ)
epsB = make_su2_structure_constant(SU2B_ADJ)
epsC = make_su2_structure_constant(SU2C_ADJ)

TA_fund = make_su2_generator(SU2A_ADJ, SU2A_FUND)
TA_trip = epsA

TB_fund = make_su2_generator(SU2B_ADJ, SU2B_FUND)
TB_trip = epsB

TC_fund = make_su2_generator(SU2C_ADJ, SU2C_FUND)
TC_trip = epsC


WA = Field(
    "WA",
    spin=1,
    self_conjugate=True,
    symbol=S("WA"),
    indices=(LORENTZ_INDEX, SU2A_ADJ),
)
WB = Field(
    "WB",
    spin=1,
    self_conjugate=True,
    symbol=S("WB"),
    indices=(LORENTZ_INDEX, SU2B_ADJ),
)
WC = Field(
    "WC",
    spin=1,
    self_conjugate=True,
    symbol=S("WC"),
    indices=(LORENTZ_INDEX, SU2C_ADJ),
)

qL = Field(
    "qL",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("qL"),
    conjugate_symbol=S("qLbar"),
    indices=(SPINOR_INDEX, SU2A_FUND, SU2C_ADJ),
)
chiL = Field(
    "chiL",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("chiL"),
    conjugate_symbol=S("chiLbar"),
    indices=(SPINOR_INDEX, SU2B_FUND, SU2C_FUND),
)
etaR = Field(
    "etaR",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("etaR"),
    conjugate_symbol=S("etaRbar"),
    indices=(SPINOR_INDEX, SU2A_ADJ, SU2B_FUND),
)
PhiDiag = Field(
    "Phi",
    spin=0,
    self_conjugate=False,
    symbol=S("PhiDiag"),
    conjugate_symbol=S("PhiDiagBar"),
    indices=(SU2A_FUND, SU2B_FUND, SU2C_ADJ),
)





SU2A = GaugeGroup(
    name="SU2A",
    abelian=False,
    coupling=gA,
    gauge_boson=WA,
    structure_constant=epsA,
    representations=(
        GaugeRepresentation(
            index=SU2A_FUND,
            generator_builder=TA_fund,
            name="doublet",
        ),
        GaugeRepresentation(
            index=SU2A_ADJ,
            generator_builder=TA_trip,
            name="triplet",
        ),
    ),
)

SU2B = GaugeGroup(
    name="SU2B",
    abelian=False,
    coupling=gB,
    gauge_boson=WB,
    structure_constant=epsB,
    representations=(
        GaugeRepresentation(
            index=SU2B_FUND,
            generator_builder=TB_fund,
            name="doublet",
        ),
        GaugeRepresentation(
            index=SU2B_ADJ,
            generator_builder=TB_trip,
            name="triplet",
        ),
    ),
)

SU2C = GaugeGroup(
    name="SU2C",
    abelian=False,
    coupling=gC,
    gauge_boson=WC,
    structure_constant=epsC,
    representations=(
        GaugeRepresentation(
            index=SU2C_FUND,
            generator_builder=TC_fund,
            name="doublet",
        ),
        GaugeRepresentation(
            index=SU2C_ADJ,
            generator_builder=TC_trip,
            name="triplet",
        ),
    ),
)

su2_triplicate_model = Model(
    gauge_groups=(SU2A, SU2B, SU2C),
    fields=(qL, chiL, etaR, PhiDiag, WA, WB, WC),
    lagrangian_decl=(
        I * qL.bar * Gamma(mu) * CovD(qL, mu)
        + I * chiL.bar * Gamma(mu) * CovD(chiL, mu)
        + I * etaR.bar * Gamma(mu) * CovD(etaR, mu)
        + CovD(PhiDiag.bar, mu) * CovD(PhiDiag, mu)
        - (Expression.num(1) / Expression.num(4))
        * FieldStrength(SU2A, mu, nu)
        * FieldStrength(SU2A, mu, nu)
        - (Expression.num(1) / Expression.num(4))
        * FieldStrength(SU2B, mu, nu)
        * FieldStrength(SU2B, mu, nu)
        - (Expression.num(1) / Expression.num(4))
        * FieldStrength(SU2C, mu, nu)
        * FieldStrength(SU2C, mu, nu)
    ),
)
su2_triplicate_lagrangian = su2_triplicate_model.lagrangian()


print("Representation assignments (SU2A, SU2B, SU2C):")
print("qL   ~ (2, 1, 3)")
print("chiL ~ (1, 2, 2)")
print("etaR ~ (3, 2, 1)")
print("Phi  ~ (2, 2, 3)")
print()

print("Expected gauge-group matching:")
print("qL   -> SU2A, SU2C")
print("chiL -> SU2B, SU2C")
print("etaR -> SU2A, SU2B")
print("Phi  -> SU2A, SU2B, SU2C")
print()


def has_vertex(*fields):
    try:
        su2_triplicate_lagrangian.feynman_rule(*fields, simplify=True)
    except ValueError:
        return False
    return True


def inspect_vertex(label, *fields):
    print(label)
    try:
        print(su2_triplicate_lagrangian.feynman_rule(*fields, simplify=True))
    except ValueError:
        print("ABSENT")
    print()


print("Presence/absence scan:")
for label, fields in (
    ("qL with WA", (qL.bar, qL, WA)),
    ("qL with WB", (qL.bar, qL, WB)),
    ("qL with WC", (qL.bar, qL, WC)),
    ("chiL with WA", (chiL.bar, chiL, WA)),
    ("chiL with WB", (chiL.bar, chiL, WB)),
    ("chiL with WC", (chiL.bar, chiL, WC)),
    ("etaR with WA", (etaR.bar, etaR, WA)),
    ("etaR with WB", (etaR.bar, etaR, WB)),
    ("etaR with WC", (etaR.bar, etaR, WC)),
    ("Phi with WA", (PhiDiag.bar, PhiDiag, WA)),
    ("Phi with WB", (PhiDiag.bar, PhiDiag, WB)),
    ("Phi with WC", (PhiDiag.bar, PhiDiag, WC)),
):
    print(f"{label}: {'present' if has_vertex(*fields) else 'absent'}")
print()

print("Representative non-zero vertices:")
inspect_vertex("qL.bar, qL, WA", qL.bar, qL, WA)
inspect_vertex("qL.bar, qL, WC", qL.bar, qL, WC)
inspect_vertex("chiL.bar, chiL, WB", chiL.bar, chiL, WB)
inspect_vertex("chiL.bar, chiL, WC", chiL.bar, chiL, WC)
inspect_vertex("etaR.bar, etaR, WA", etaR.bar, etaR, WA)
inspect_vertex("etaR.bar, etaR, WB", etaR.bar, etaR, WB)
inspect_vertex("Phi.bar, Phi, WA", PhiDiag.bar, PhiDiag, WA)
inspect_vertex("Phi.bar, Phi, WB", PhiDiag.bar, PhiDiag, WB)
inspect_vertex("Phi.bar, Phi, WC", PhiDiag.bar, PhiDiag, WC)

print("Mixed scalar contact terms from CovD(Phi.bar, mu) CovD(Phi, mu):")
inspect_vertex("Phi.bar, Phi, WA, WB", PhiDiag.bar, PhiDiag, WA, WB)
inspect_vertex("Phi.bar, Phi, WA, WC", PhiDiag.bar, PhiDiag, WA, WC)
inspect_vertex("Phi.bar, Phi, WB, WC", PhiDiag.bar, PhiDiag, WB, WC)


Representation assignments (SU2A, SU2B, SU2C):
qL   ~ (2, 1, 3)
chiL ~ (1, 2, 2)
etaR ~ (3, 2, 1)
Phi  ~ (2, 2, 3)

Expected gauge-group matching:
qL   -> SU2A, SU2C
chiL -> SU2B, SU2C
etaR -> SU2A, SU2B
Phi  -> SU2A, SU2B, SU2C

Presence/absence scan:
qL with WA: present
qL with WB: absent
qL with WC: present
chiL with WA: absent
chiL with WB: present
chiL with WC: present
etaR with WA: present
etaR with WB: present
etaR with WC: absent
Phi with WA: present
Phi with WB: present
Phi with WC: present

Representative non-zero vertices:
qL.bar, qL, WA
1𝑖*gA*g(coad(3, ac1),coad(3, ac2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(3, aa3),cof(2, a1),cof(2, a2))

qL.bar, qL, WC
1𝑖*gC*g(cof(2, a1),cof(2, a2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*f(coad(3, ac3),coad(3, ac1),coad(3, ac2))

chiL.bar, chiL, WB
1𝑖*gB*g(cof(2, c1),cof(2, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(3, ab3),cof(2, b1),cof(2, b2))

chiL.bar, chiL, WC
1𝑖*gC*g(cof(2, b1),cof(2, b2))*gamma(bis(4, i1),

This example is useful because the field-to-gauge-group matching cannot rely only on the group type SU(2). It must use the specific representation index attached to each gauge group.

## 6. Vertices, Feynman Rules, and Discovery

`feynman_rule(...)` has two user-facing modes:

- with explicit fields, it returns one momentum-space vertex expression;
- with no fields, it returns a dictionary mapping vertex signatures to expressions.

The output expressions contain coupling constants, momentum dependence, open index structure, and the overall momentum-conservation factor. The order of explicit fields fixes the displayed momenta and automatically assigned open indices.

In [ ]:
p_in, p_out, k = S("p_in", "p_out", "k")

show(
    "QED vertex with default momenta",
    qed_lagrangian.feynman_rule(Electron.bar, Electron, Photon, simplify=True),
)
show(
    "QED vertex with custom momenta",
    qed_lagrangian.feynman_rule(
        Electron.bar,
        Electron,
        Photon,
        momenta=[p_in, p_out, k],
        simplify=True,
    ),
)


QED vertex with default momenta
1𝑖*e*Qe*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))

QED vertex with custom momenta
1𝑖*e*Qe*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))



The structure is easy to read once the conventions are clear:

- `Delta(...)` is the overall momentum-conservation factor,
- `gamma(...)` is the open gamma-matrix structure,
- `t(...)` and `f(...)` are generator and structure-constant tensors when non-abelian groups are involved,
- `g(...)` is the metric tensor,
- `pcomp(p, mu)` is the component of momentum `p` along index `mu`.

Using `momenta=[...]` is useful when you want the displayed momentum labels to match a paper or calculation.

### High-Level Output Controls

The high-level `feynman_rule(...)` API now exposes the most common output-policy controls directly:

- `include_delta=True` keeps the overall momentum-conservation factor;
- `strip_externals=False` keeps the external-wavefunction layer instead of the stripped default output;
- `simplify_gamma=True` forwards into the optional gamma-chain simplification pass.

For `strip_externals=False`, use `simplify=False` if you want to see the preserved external factors explicitly before later cleanup contracts them away.


In [ ]:
show(
    "QED vertex without Delta",
    qed_lagrangian.feynman_rule(
        Electron.bar, Electron, Photon,
        simplify=False,
        include_delta=False,
    ),
)

show(
    "Yukawa vertex with external wavefunctions kept",
    fermion_lagrangian.feynman_rule(
        Psi.bar, Psi, Phi,
        simplify=False,
        strip_externals=False,
    ),
)

g_gamma = S("g_gamma")
gamma_demo_lagrangian = Lagrangian(
    g_gamma * Psi.bar * Gamma(mu) * Gamma(nu) * Psi
    + g_gamma * Psi.bar * Gamma(nu) * Gamma(mu) * Psi
)

show(
    "Gamma-chain before simplify_gamma",
    gamma_demo_lagrangian.feynman_rule(Psi.bar, Psi, simplify=True),
)

show(
    "Gamma-chain with simplify_gamma=True",
    gamma_demo_lagrangian.feynman_rule(
        Psi.bar, Psi,
        simplify=True,
        simplify_gamma=True,
    ),
)


QED vertex without Delta
1𝑖*e*Qe*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*delta(e,e)*delta(A,A)*delta(ebar,ebar)

Yukawa vertex with external wavefunctions kept
1𝑖*y*exp(-1𝑖*Dot(q1+q2+q3,x_))*g(bis(4, i1),bis(4, i2))*delta(phi,phi)*delta(psi,psi)*delta(psibar,psibar)*U(phi,q3)

Gamma-chain before simplify_gamma
1𝑖*g_gamma*gamma(bis(4, i1),bis(4, spinor_decl_3),mink(4, mu))*gamma(bis(4, spinor_decl_3),bis(4, i2),mink(4, nu))+1𝑖*g_gamma*gamma(bis(4, i1),bis(4, spinor_decl_3),mink(4, nu))*gamma(bis(4, spinor_decl_3),bis(4, i2),mink(4, mu))

Gamma-chain with simplify_gamma=True
2𝑖*g_gamma*g(mink(4, mu),mink(4, nu))*g(bis(4, i1),bis(4, i2))



### Enumerating Vertices in a Larger Gauge Sector

Calling `feynman_rule()` with no field arguments asks the compiled Lagrangian for every available vertex signature. By default, the result is a dictionary keyed by field-name tuples, for example `("ghW.bar", "W", "ghW")`. If you need the actual `Field` / `Field.bar` objects as keys, use `key_format="fields"`.

For a larger model, use `simplify=False` first so discovery stays lightweight. The `show(...)` helper below prints each discovered signature followed by the corresponding expression.

The model below contains SU(2) x U(1) matter covariant derivatives, both field-strength terms, both gauge-fixing terms, and the non-abelian SU(2) ghost sector. The abelian U(1) gauge-fixing term is included; the ordinary ghost declaration is only used for SU(2), where the non-abelian ghost-gauge coupling is nontrivial.

In [ ]:
WeakGhost = Field(
    "ghW",
    spin=0,
    kind="ghost",
    self_conjugate=False,
    symbol=S("ghW"),
    conjugate_symbol=S("ghWbar"),
    indices=(WEAK_ADJ_INDEX,),
)

SU2L_GHOST = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=g2,
    gauge_boson=W,
    ghost_field=WeakGhost,
    structure_constant=weak_structure_constant,
    representations=(WEAK_DOUBLET_REP,),
)

ew_gauge_fixed_model = Model(
    gauge_groups=(SU2L_GHOST, U1Y),
    fields=(LDoublet, ERight, Higgs, W, B, WeakGhost),
    lagrangian_decl=(
        I * LDoublet.bar * Gamma(mu) * CovD(LDoublet, mu)
        + I * ERight.bar * Gamma(mu) * CovD(ERight, mu)
        + CovD(Higgs.bar, mu) * CovD(Higgs, mu)
        + (
            -(Expression.num(1) / Expression.num(4))
            * FieldStrength(SU2L_GHOST, mu, nu)
            * FieldStrength(SU2L_GHOST, mu, nu)
        )
        + (
            -(Expression.num(1) / Expression.num(4))
            * FieldStrength(U1Y, mu, nu)
            * FieldStrength(U1Y, mu, nu)
        )
        + GaugeFixing(SU2L_GHOST, xi=xi)
        + GaugeFixing(U1Y, xi=xi)
        + GhostLagrangian(SU2L_GHOST)
    ),
)

ew_gauge_fixed_lagrangian = ew_gauge_fixed_model.lagrangian()
ew_gauge_fixed_rules = ew_gauge_fixed_lagrangian.feynman_rule(simplify=False)

print("Compiled interaction terms:", len(ew_gauge_fixed_lagrangian.terms))
show(
    "Gauge-fixed SU(2) x U(1) vertices discovered by feynman_rule()",
    ew_gauge_fixed_rules,
)

ew_gauge_fixed_rules_by_field = ew_gauge_fixed_lagrangian.feynman_rule(
    simplify=False,
    key_format="fields",
)
print("First object-keyed signature:", next(iter(ew_gauge_fixed_rules_by_field)))


Compiled interaction terms: 24
Gauge-fixed SU(2) x U(1) vertices discovered by feynman_rule()
17 vertex signature(s)

Vertex: ('L.bar', 'L', 'W')
Rule: 1𝑖*g2*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(3, aw3),cof(2, w1),cof(2, w2))*delta(W,W)*delta(L,L)*delta(Lbar,Lbar)

Vertex: ('L.bar', 'L', 'B')
Rule: 1𝑖*g1*YL*g(cof(2, w1),cof(2, w2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*delta(B,B)*delta(L,L)*delta(Lbar,Lbar)

Vertex: ('L.bar', 'L')
Rule: 1𝑖*g(cof(2, w1),cof(2, w2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu1_int))*pcomp(q2,mu1_int)*delta(L,L)*delta(Lbar,Lbar)

Vertex: ('ER.bar', 'ER', 'B')
Rule: 1𝑖*g1*YR*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*delta(B,B)*delta(ER,ER)*delta(ERbar,ERbar)

Vertex: ('ER.bar', 'ER')
Rule: 1𝑖*gamma(bis(4, i1),bis(4, i2),mink(4, mu1_int))*pcomp(q2,mu1_int)*delta(ER,ER)*delta(ERbar,ERbar)

Vertex: ('H.bar', 'H', 'W')
Rule: -1𝑖*g2*t(coad(3, aw3),cof(2, w1),cof(2, w2))*pcomp(q1,mu)*delta(W,W)*delta(H,H)*delta(Hdag,Hdag)+1𝑖*g2*t(coad(3, aw3),cof(2, w

### Structured Vertex Reports and Filters

The compiled Lagrangian now exposes a lightweight reporting layer on top of vertex discovery. This is useful when you want to inspect what a model contains **before** extracting or simplifying every rule.

Current capabilities:

- `vertex_signatures()` lists grouped compiled signatures in deterministic order;
- `vertex_signatures(arity=...)` keeps only vertices of a chosen size;
- `vertex_signatures(signature=(...))` matches one exact field-content signature;
- `vertex_signatures(contains_fields=(...))` finds every signature containing a requested field multiset;
- `vertex_report(...)` returns the same filtered signatures together with total and matched counts.

The filters use the same field matching conventions as `feynman_rule(...)`. In particular, exact signature matching is by **field content**, not by display order.


In [ ]:
# QCD: list grouped compiled signatures without extracting every vertex
for signature in qcd_lagrangian.vertex_signatures():
    print(signature.names, "| arity=", signature.arity, "| term_count=", signature.term_count)

print("=" * 72)

# Exact field-content filter (order-independent)
qcd_exact = qcd_lagrangian.vertex_report(signature=(Gluon, Quark.bar, Quark))
print("Exact QCD report:", qcd_exact.matched_signatures, "match of", qcd_exact.total_signatures, "total signatures")
for signature in qcd_exact.signatures:
    print("  ", signature.names, "| term_count=", signature.term_count)

print("=" * 72)

# Larger model: filter by arity and by contained fields
ew_three_point = ew_gauge_fixed_lagrangian.vertex_report(arity=3)
print("Gauge-fixed EW 3-point signatures:", ew_three_point.matched_signatures, "of", ew_three_point.total_signatures)
for signature in ew_three_point.signatures[:6]:
    print("  ", signature.names)

print("=" * 72)

ghost_related = ew_gauge_fixed_lagrangian.vertex_signatures(
    contains_fields=(WeakGhost.bar, WeakGhost),
)
print("Ghost-related signatures:")
for signature in ghost_related:
    print("  ", signature.names)


('q.bar', 'q') | arity= 2 | term_count= 1
('q.bar', 'q', 'G') | arity= 3 | term_count= 1
Exact QCD report: 1 match of 2 total signatures
   ('q.bar', 'q', 'G') | term_count= 1
Gauge-fixed EW 3-point signatures: 7 of 17
   ('ER.bar', 'ER', 'B')
   ('H.bar', 'H', 'B')
   ('H.bar', 'H', 'W')
   ('L.bar', 'L', 'B')
   ('L.bar', 'L', 'W')
   ('W', 'W', 'W')
Ghost-related signatures:
   ('ghW.bar', 'ghW')
   ('ghW.bar', 'W', 'ghW')


### Model Validation Diagnostics

`Model.validate()` provides a lightweight pre-extraction diagnostic pass for gauge-sector consistency. It does **not** compile or modify vertices; it only reports issues already visible from model metadata and declared source terms.

Current checks include undeclared gauge-group references in gauge-fixing / ghost declarations, abelian ghost-sector requests, missing non-abelian `structure_constant` builders, and missing `ghost_field` declarations.


In [ ]:
# Valid model: gauge-sector diagnostics should come back clean
valid_report = ew_gauge_fixed_model.validate()
print('Valid EW gauge-sector report ok:', valid_report.ok)
print('Issues:', valid_report.issues)

print('=' * 72)

# Intentionally inconsistent model: abelian ghost sector
BadGhost = Field(
    'ghA_diag',
    spin=0,
    kind='ghost',
    self_conjugate=False,
    symbol=S('ghA_diag'),
    conjugate_symbol=S('ghAbar_diag'),
)

U1Diag = GaugeGroup(
    name='U1Diag',
    abelian=True,
    coupling=S('eDiag'),
    gauge_boson=B,
    ghost_field=BadGhost,
    charge='Y',
)

bad_model = Model(
    gauge_groups=(U1Diag,),
    fields=(B, BadGhost),
    lagrangian_decl=GhostLagrangian(U1Diag),
)

bad_report = bad_model.validate()
print('Bad gauge-sector report ok:', bad_report.ok)
for issue in bad_report.issues:
    print(' -', issue.code, '|', issue.message)


Valid EW gauge-sector report ok: True
Issues: ()
Bad gauge-sector report ok: False
 - abelian_ghost_sector | Ghost validation only supports non-abelian gauge groups; got 'U1Diag'.


## 7. Capability Map

The notebook demonstrates the following current capabilities:

- field declarations for real scalars, complex scalars, Dirac fermions, gauge bosons, and non-abelian ghosts;
- local polynomial interactions, Yukawa-like fermion couplings, and derivative operators built from `PartialD(...)`;
- abelian and non-abelian gauge currents via `CovD(...)`;
- pure gauge sectors via `FieldStrength(...)`, including Yang-Mills cubic and quartic self-interactions;
- gauge fixing via `GaugeFixing(...)` and ordinary non-abelian Faddeev-Popov ghosts via `GhostLagrangian(...)`;
- mixed gauge sectors such as SU(2) x U(1), including matter currents and scalar gauge-contact terms;
- targeted vertex extraction through `feynman_rule(Field1, Field2, ...)`;
- full vertex discovery through zero-argument `feynman_rule()`, keyed by readable field-name tuples by default;
- object-keyed discovery through `feynman_rule(key_format="fields")` when programmatic lookup by `Field` objects is needed;
- structured signature discovery and reporting through `vertex_signatures(...)` and `vertex_report(...)`, including arity and field-content filters.
- structured gauge-sector diagnostics through `Model.validate()`, with read-only issue reporting before compilation or extraction.

A practical rule of thumb is: use explicit fields when you know the process, and use zero-argument `feynman_rule()` when you want to inspect what the compiled Lagrangian contains.

## 8. Practical Notes and Conventions

- Use `Field.bar` for conjugated non-self-conjugate fields in both declarations and explicit `feynman_rule(...)` calls.
- Use `Lagrangian(...)` for local terms that do not require gauge metadata.
- Use `Model(..., lagrangian_decl=...)` when the declaration involves `CovD(...)`, `FieldStrength(...)`, `GaugeFixing(...)`, or `GhostLagrangian(...)`.
- Default external momenta are named `q1`, `q2`, `q3`, ... in the order you pass the external fields.
- By default the returned expression omits the universal `(2*pi)^d Delta(sum p)` factor.
- The order of fields in explicit `feynman_rule(...)` calls fixes the displayed momentum labels and open indices, so use a consistent ordering when comparing vertices.
- For non-abelian ghosts, the gauge group must know its ghost field through `ghost_field=...`.
- If a field carries the same index type more than once, you may need `GaugeRepresentation(slot=...)` or `slot_policy='sum'` to resolve the intended action.
- Zero-argument `feynman_rule()` returns name-tuple keys such as `("ghW.bar", "W", "ghW")`; use `key_format="fields"` if you need `Field` / `Field.bar` objects as keys.

- Use `vertex_signatures(...)` or `vertex_report(...)` when you want to inspect grouped compiled signatures without extracting every expression first.

- `signature=(...)` matching in `vertex_signatures(...)` / `vertex_report(...)` is by field content, not by display order.
- For large models, start with `simplify=False` when enumerating all vertices, then extract important vertices explicitly with `simplify=True`.


In [ ]:
# Model: QCD current-current operator with covariant derivatives
qcd_current_current_model = Model(
    gauge_groups=(SU3C,),
    fields=(Quark, Gluon),
    lagrangian_decl=(
        I
        * Quark.bar
        * Gamma(mu)
        * CovD(Quark, mu)
        * Quark.bar
        * Gamma(nu)
        * CovD(Quark, nu)
    ),
)

# Build Lagrangian
qcd_current_current_lagrangian = qcd_current_current_model.lagrangian()

# Compute all vertices
qcd_current_current_vertices = qcd_current_current_lagrangian.feynman_rule(
    simplify=True,
)

# Display
show(
    "QCD current-current operator vertices (q̄γ·Dq)(q̄γ·Dq)",
    qcd_current_current_vertices,
)

ValueError: Index label 'nu' is used with incompatible index types 'Lorentz' and 'Lorentz' in one monomial (Gamma; G slot 1).

In [ ]:

qcd_4q_1g_vertex = qcd_current_current_lagrangian.feynman_rule(
    Quark.bar, Quark, Quark.bar, Quark, Gluon,
    simplify=True,
)
print("QCD current-current-gluon vertex Gamma(q.bar, q, q.bar, q, G)\n", qcd_4q_1g_vertex)

QCD current-current-gluon vertex Gamma(q.bar, q, q.bar, q, G)
 2*gs*g(cof(3, c3),cof(3, c4))*gamma(bis(4, i1),bis(4, i2),mink(4, mu5))*gamma(bis(4, i3),bis(4, i4),mink(4, mu1_int))*t(coad(8, a5),cof(3, c1),cof(3, c2))*pcomp(q4,mu1_int)+2*gs*g(cof(3, c3),cof(3, c2))*gamma(bis(4, i3),bis(4, i2),mink(4, mu1_int))*gamma(bis(4, i1),bis(4, i4),mink(4, mu5))*t(coad(8, a5),cof(3, c1),cof(3, c4))*pcomp(q2,mu1_int)+2*gs*g(cof(3, c4),cof(3, c1))*gamma(bis(4, i1),bis(4, i4),mink(4, mu1_int))*gamma(bis(4, i3),bis(4, i2),mink(4, mu5))*t(coad(8, a5),cof(3, c3),cof(3, c2))*pcomp(q4,mu1_int)+2*gs*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu1_int))*gamma(bis(4, i3),bis(4, i4),mink(4, mu5))*t(coad(8, a5),cof(3, c3),cof(3, c4))*pcomp(q2,mu1_int)
